# geemap.ee_to_xarray xee v0.1.0 Migration Test

This notebook validates that geemap.ee_to_xarray works with xee v0.1.0+ and remains backward compatible with the legacy geemap call pattern.

## 1) Imports and version check

In [ ]:
import ee
import shapely
import xarray as xr
import xee
from xee import helpers
import geemap

print(f"geemap: {geemap.__version__}")
print(f"xee: {xee.__version__}")
print(f"xarray: {xr.__version__}")

In [ ]:
import importlib
import inspect
import geemap.common as geemap_common

# Ensure notebook uses the latest local code while iterating on geemap/common.py
geemap_common = importlib.reload(geemap_common)
geemap.ee_to_xarray = geemap_common.ee_to_xarray

src = inspect.getsource(geemap_common.ee_to_xarray)
print("Using geemap.common from:", geemap_common.__file__)
print("Auto grid fallback present:", "helpers.extract_grid_params" in src)
print(
    "Explicit grid detection fix present:",
    'explicit_grid_crs = kwargs.get("crs", crs)' in src,
)
print("Robust geometry conversion present:", "shapely_shape(geometry.getInfo())" in src)
print("Stale import check passed")

## 2) Earth Engine initialization

If this is your first run, authenticate first by uncommenting ee.Authenticate().

In [ ]:
# ee.Authenticate()
ee.Initialize(project="ee-braaten")
print("Earth Engine initialized")

## 3) Legacy geemap API call (the bug path)

This is the call pattern reported in issue #2780. It should now work with xee v0.1.0+.

In [ ]:
ds_legacy = geemap.ee_to_xarray(
    "ECMWF/ERA5_LAND/HOURLY",
    crs="EPSG:4326",
    scale=0.25,
    n_images=5,
    ee_initialize=False,
)
ds_legacy

## 4) Native xee v0.1.0 style reference call

This is the explicit grid parameter flow using xee helpers.

Use the same target grid as Part 3 (EPSG:4326, scale=0.25, global extent) for an equivalent comparison.

In [ ]:
ic = ee.ImageCollection("ECMWF/ERA5_LAND/HOURLY")
grid_params = helpers.fit_geometry(
    geometry=shapely.geometry.box(-180, -90, 180, 90),
    grid_crs="EPSG:4326",
    grid_scale=(0.25, -0.25),
)

ds_native = xr.open_dataset(
    ic,
    engine="ee",
    n_images=5,
    **grid_params,
)
ds_native

## 5) Quick comparison checks

In [ ]:
print("legacy dims:", ds_legacy.dims)
print("native dims:", ds_native.dims)

legacy_vars = set(ds_legacy.data_vars)
native_vars = set(ds_native.data_vars)

print("legacy var count:", len(legacy_vars))
print("native var count:", len(native_vars))
print("overlap var count:", len(legacy_vars.intersection(native_vars)))

## 6) Expanded test matrix setup

The cells below cover additional legacy and new-style parameter combinations to validate backward compatibility and new xee-enabled usage patterns.

In [ ]:
import matplotlib.pyplot as plt

results = {}


def run_case(name, fn, expected_fail=False):
    try:
        ds = fn()
        if expected_fail:
            results[name] = {
                "ok": False,
                "expected_fail": True,
                "error": "Expected failure but call succeeded",
            }
            print(f"UNEXPECTED PASS: {name} (this case was marked expected_fail)")
            return ds

        results[name] = {
            "ok": True,
            "dims": dict(ds.sizes),
            "vars": len(ds.data_vars),
            "expected_fail": False,
        }
        print(f"PASS: {name} -> dims={dict(ds.sizes)} vars={len(ds.data_vars)}")
        return ds
    except Exception as e:
        if expected_fail:
            results[name] = {
                "ok": True,
                "dims": None,
                "vars": None,
                "expected_fail": True,
                "error": str(e),
            }
            print(f"EXPECTED FAIL: {name} -> {e}")
            return None

        results[name] = {"ok": False, "expected_fail": False, "error": str(e)}
        print(f"FAIL: {name} -> {e}")
        return None


def first_data_var(ds):
    return list(ds.data_vars)[0]


def show_first_map(ds, title):
    var = first_data_var(ds)
    da = ds[var]
    if "time" in da.dims:
        da = da.isel(time=0)
    da.plot(figsize=(8, 4))
    plt.title(f"{title} | var={var}")
    plt.tight_layout()
    plt.show()

In [ ]:
ic = ee.ImageCollection("ECMWF/ERA5_LAND/HOURLY")
img = ee.Image("LANDSAT/LC08/C02/T1_TOA/LC08_044034_20140318")
region = ee.Geometry.Rectangle([113.33, -43.63, 153.56, -10.66])
projection = ic.first().select(0).projection()

native_source_grid = helpers.extract_grid_params(ic)
native_scaled_grid = helpers.fit_geometry(
    geometry=shapely.geometry.box(-180, -90, 180, 90),
    grid_crs="EPSG:4326",
    grid_scale=(0.25, -0.25),
)

print("Prepared objects for tests.")

## 7) Legacy-style usage matrix (backward compatibility)

These cases reflect common prior usage patterns that should continue to work.

In [ ]:
ds_case_a = run_case(
    "A: asset id only",
    lambda: geemap.ee_to_xarray(
        "ECMWF/ERA5_LAND/HOURLY", n_images=3, ee_initialize=False
    ),
)

ds_case_b = run_case(
    "B: asset id + crs + scale",
    lambda: geemap.ee_to_xarray(
        "ECMWF/ERA5_LAND/HOURLY",
        crs="EPSG:4326",
        scale=0.25,
        n_images=3,
        ee_initialize=False,
    ),
)

ds_case_c = run_case(
    "C: asset id + crs + scale + geometry",
    lambda: geemap.ee_to_xarray(
        "ECMWF/ERA5_LAND/HOURLY",
        crs="EPSG:4326",
        scale=0.25,
        geometry=region,
        n_images=3,
        ee_initialize=False,
    ),
)

ds_case_c2 = run_case(
    "C2: asset id + geometry only (legacy)",
    lambda: geemap.ee_to_xarray(
        "ECMWF/ERA5_LAND/HOURLY",
        geometry=region,
        n_images=3,
        ee_initialize=False,
    ),
)

ds_case_d = run_case(
    "D: image collection object",
    lambda: geemap.ee_to_xarray(ic, n_images=3, ee_initialize=False),
)

ds_case_e = run_case(
    "E: image object",
    lambda: geemap.ee_to_xarray(img, ee_initialize=False),
)

In [ ]:
ds_case_f = run_case(
    "F: projection parameter",
    lambda: geemap.ee_to_xarray(
        ic,
        projection=projection,
        n_images=3,
        ee_initialize=False,
    ),
)

# xarray.open_mfdataset + xee does not currently provide robust concat behavior
# for repeated EE assets without explicit concat coordinates.
# Keep this case to document current limitation, but mark as expected failure.
ds_case_g = run_case(
    "G: list of asset ids (known limitation)",
    lambda: geemap.ee_to_xarray(
        ["ECMWF/ERA5_LAND/HOURLY", "ECMWF/ERA5_LAND/HOURLY"],
        n_images=2,
        ee_initialize=False,
    ),
    expected_fail=True,
)

ds_case_h = run_case(
    "H: drop_variables",
    lambda: geemap.ee_to_xarray(
        "ECMWF/ERA5_LAND/HOURLY",
        n_images=3,
        drop_variables=["temperature_2m"],
        ee_initialize=False,
    ),
)

ds_case_i = run_case(
    "I: request_byte_limit passthrough",
    lambda: geemap.ee_to_xarray(
        "ECMWF/ERA5_LAND/HOURLY",
        n_images=3,
        request_byte_limit=16 * 1024 * 1024,
        ee_initialize=False,
    ),
)

## 8) New xee-style usage through geemap

These cases intentionally use xee v0.1.0 grid parameters passed via geemap.ee_to_xarray kwargs.

In [ ]:
ds_case_j = run_case(
    "J: source-native grid via extract_grid_params",
    lambda: geemap.ee_to_xarray(
        ic,
        n_images=3,
        ee_initialize=False,
        **native_source_grid,
    ),
)

ds_case_k = run_case(
    "K: explicit custom grid via fit_geometry",
    lambda: geemap.ee_to_xarray(
        ic,
        n_images=3,
        ee_initialize=False,
        **native_scaled_grid,
    ),
)

ds_case_l = run_case(
    "L: direct grid kwargs (crs/crs_transform/shape_2d)",
    lambda: geemap.ee_to_xarray(
        ic,
        n_images=3,
        ee_initialize=False,
        crs=native_scaled_grid["crs"],
        crs_transform=native_scaled_grid["crs_transform"],
        shape_2d=native_scaled_grid["shape_2d"],
    ),
)

## 9) Plot checks with xarray + matplotlib

These cells validate that resulting datasets are directly plottable and have expected dimension ordering.

In [ ]:
show_first_map(ds_legacy, "Legacy geemap path")
show_first_map(ds_native, "Native xee equivalent grid")
show_first_map(ds_case_c2, "Geometry-only legacy path")
show_first_map(ds_case_d, "ImageCollection object path")

In [ ]:
var = first_data_var(ds_legacy)
spatial_mean = ds_legacy[var].mean(
    dim=[d for d in ["y", "x"] if d in ds_legacy[var].dims]
)

spatial_mean.plot(marker="o", figsize=(8, 3))
plt.title(f"Time series (spatial mean) for {var}")
plt.tight_layout()
plt.show()

## 10) Result summary

This compact table helps confirm which parameter combinations passed and what output shape they produced.

In [ ]:
import pandas as pd

rows = []
for name, item in results.items():
    rows.append(
        {
            "case": name,
            "ok": item.get("ok", False),
            "expected_fail": item.get("expected_fail", False),
            "dims": item.get("dims", None),
            "vars": item.get("vars", None),
            "error": item.get("error", None),
        }
    )

summary_df = pd.DataFrame(rows).sort_values("case").reset_index(drop=True)
summary_df